# AIML339 Galaxy Morphology Classification Comparison Project

donaldseth - 300665568

In [70]:
# Imports

import pandas as pd
import numpy as np

import random

from zipfile import ZipFile
from PIL import Image
from io import BytesIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

In [71]:
# Set random seeds for reproducible experiments

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Random seeds set to 42")

Random seeds set to 42


In [27]:
# Paths

LABELS_PATH = "training_solutions_rev1.csv"
IMAGES_PATH = "images_training_rev1.zip"

In [28]:
# Load Labels

labels = pd.read_csv(LABELS_PATH)

print("Total galaxies:", len(labels))
print("Number of classes:", len(labels.columns) - 1)

Total galaxies: 61578
Number of classes: 37


In [29]:
# Labels Subset

N_GALAXIES = 5000

labels_subset = labels.sample(
    n=N_GALAXIES,
    random_state=42
).copy()

labels_subset["filename"] = (
    labels_subset["GalaxyID"].astype(str)
)

labels_subset.head()

,GalaxyID,Class1.1,Class1.2,Class1.3,Class2.1,Class2.2,Class3.1,Class3.2,Class4.1,Class4.2,...,Class10.1,Class10.2,Class10.3,Class11.1,Class11.2,Class11.3,Class11.4,Class11.5,Class11.6,filename
30888,553402,0.018765,0.955090,0.026146,0.078768,0.876322,0.118995,0.757327,0.590601,0.285720,...,0.000000,0.136383,0.454219,0.193254,0.289966,0.000000,0.0,0.000000,0.107381,553402
23964,449744,0.806705,0.140380,0.052915,0.000000,0.140380,0.000000,0.140380,0.000000,0.140380,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,449744
53475,879640,0.662354,0.337646,0.000000,0.197076,0.140570,0.000000,0.140570,0.044101,0.096468,...,0.044101,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.044101,879640
16284,340485,0.092952,0.907048,0.000000,0.058283,0.848765,0.042052,0.806713,0.642360,0.206404,...,0.496498,0.134098,0.011765,0.000000,0.373704,0.010232,0.0,0.013647,0.244777,340485
60574,985545,0.144641,0.848689,0.006670,0.028017,0.820672,0.484659,0.336014,0.267777,0.552895,...,0.000000,0.219558,0.048219,0.000000,0.195449,0.000000,0.0,0.000000,0.072328,985545


In [30]:
# Collect from .zip file

with ZipFile(IMAGES_PATH, "r") as zip_file:
    
    images = []

    for galaxy_id in labels_subset["filename"]:
        filename = f"images_training_rev1/{galaxy_id}.jpg"
        image_data = zip_file.read(filename)

        image = Image.open(BytesIO(image_data))
        images.append(image.copy())

print("Images collected:", len(images))

Images collected: 5000


In [31]:
print(labels_subset.columns.tolist())

['GalaxyID', 'Class1.1', 'Class1.2', 'Class1.3', 'Class2.1', 'Class2.2', 'Class3.1', 'Class3.2', 'Class4.1', 'Class4.2', 'Class5.1', 'Class5.2', 'Class5.3', 'Class5.4', 'Class6.1', 'Class6.2', 'Class7.1', 'Class7.2', 'Class7.3', 'Class8.1', 'Class8.2', 'Class8.3', 'Class8.4', 'Class8.5', 'Class8.6', 'Class8.7', 'Class9.1', 'Class9.2', 'Class9.3', 'Class10.1', 'Class10.2', 'Class10.3', 'Class11.1', 'Class11.2', 'Class11.3', 'Class11.4', 'Class11.5', 'Class11.6', 'filename']


In [32]:
# Show the response probabilities for the first two Galaxy Zoo tasks
columns = [
    "Class1.1", "Class1.2", "Class1.3",
    "Class2.1", "Class2.2"
]

print(labels_subset[columns].head(10))

       Class1.1  Class1.2  Class1.3  Class2.1  Class2.2
30888  0.018765  0.955090  0.026146  0.078768  0.876322
23964  0.806705  0.140380  0.052915  0.000000  0.140380
53475  0.662354  0.337646  0.000000  0.197076  0.140570
16284  0.092952  0.907048  0.000000  0.058283  0.848765
60574  0.144641  0.848689  0.006670  0.028017  0.820672
17962  0.553336  0.374126  0.072539  0.000000  0.374126
19482  0.529935  0.379124  0.090940  0.000000  0.379124
24973  0.518825  0.456786  0.024389  0.029735  0.427051
59184  0.130019  0.869981  0.000000  0.000000  0.869981
32165  0.438559  0.512630  0.048810  0.068876  0.443754


In [62]:
# Keep galaxies whose most likely Task 01 response is not star/artifact
task1_columns = ["Class1.1", "Class1.2", "Class1.3"]

task1_labels = labels_subset[task1_columns].idxmax(axis=1)

classification_data = labels_subset[
    task1_labels != "Class1.3"
].copy()

# The three final classes follow our simplified decision tree
class_columns = ["Class1.1", "Class2.1", "Class2.2"]

class_names = [
    "Smooth",
    "Featured edge-on",
    "Featured not edge-on"
]

# Assign labels by following the Galaxy Zoo decision hierarchy:
# Task 01: smooth vs featured
# Task 02: if featured, edge-on vs not edge-on

classification_data["target"] = np.where(
    classification_data["Class1.1"] >= classification_data["Class1.2"],
    0,
    np.where(
        classification_data["Class2.1"] >= classification_data["Class2.2"],
        1,
        2
    )
)

# Check how many galaxies belong to each class
print("Total galaxies:", len(classification_data))

print(
    classification_data["target"]
    .value_counts()
    .sort_index()
    .rename(index=dict(enumerate(class_names)))
)

Total galaxies: 4993
target
Smooth                  2147
Featured edge-on         540
Featured not edge-on    2306
Name: count, dtype: int64


In [63]:
# Check the corrected three-class label distribution
print("Total galaxies:", len(classification_data))

for class_id, class_name in enumerate(class_names):
    count = (classification_data["target"] == class_id).sum()
    print(f"{class_name}: {count}")

Total galaxies: 4993
Smooth: 2147
Featured edge-on: 540
Featured not edge-on: 2306


Dataset splitting: The initial prototype used a random 80/20 split followed by a training/validation split. For the final three-class experiments, stratified splitting is used to preserve the class distribution across training, validation and test sets.

In [64]:
# Split the corrected dataset into training, validation and test sets
# Stratification preserves the proportions of the three galaxy classes

train_val, test = train_test_split(
    classification_data,
    test_size=0.20,
    random_state=42,
    stratify=classification_data["target"]
)

train, validation = train_test_split(
    train_val,
    test_size=0.20,
    random_state=42,
    stratify=train_val["target"]
)

# Check the number of images and class distribution in each split
for name, split in [
    ("Training", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"{name}: {len(split)} images")
    print(split["target"].value_counts().sort_index().to_dict())

Training: 3195 images
{0: 1373, 1: 346, 2: 1476}
Validation: 799 images
{0: 344, 1: 86, 2: 369}
Test: 999 images
{0: 430, 1: 108, 2: 461}


In [36]:
# Image Transformation

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# For greyscale:

#transform = transforms.Compose([
#    transforms.Grayscale(num_output_channels=1),
#    transforms.Resize((64, 64)),
#    transforms.ToTensor()
#])

In [37]:
class GalaxyDataset(Dataset):
    def __init__(self, data, transform=None):
        # Store the dataset and image transformations
        self.data = data.reset_index(drop=True)
        self.transform = transform

        # Open the Galaxy Zoo image archive
        self.zip_file = ZipFile(IMAGES_PATH, "r")

    def __len__(self):
        # Return the number of galaxies in the dataset
        return len(self.data)

    def __getitem__(self, index):
        # Get the galaxy information for this row
        row = self.data.iloc[index]

        # Locate and read the image inside the ZIP archive
        filename = f"images_training_rev1/{row['filename']}.jpg"
        image_data = self.zip_file.read(filename)

        # Convert the image to RGB
        image = Image.open(BytesIO(image_data)).convert("RGB")

        # Resize and convert the image into a tensor
        if self.transform:
            image = self.transform(image)

        # Return the new three-class label as an integer
        label = torch.tensor(int(row["target"]), dtype=torch.long)

        return image, label

In [65]:
# Dataset Creation

train_dataset = GalaxyDataset(train, transform)
validation_dataset = GalaxyDataset(validation, transform)
test_dataset = GalaxyDataset(test, transform)

print("Datasets created")

Datasets created


In [66]:
# DataLoaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [67]:
images, labels_batch = next(iter(train_loader))

print("Image shape:", images.shape)
print("Label shape:", labels_batch.shape)

Image shape: torch.Size([32, 3, 64, 64])
Label shape: torch.Size([32])


In [68]:
# Confirm that each dataset uses the corrected class labels
for name, dataset in [
    ("Training", train_dataset),
    ("Validation", validation_dataset),
    ("Test", test_dataset)
]:
    print(name, dataset.data["target"].value_counts().sort_index().to_dict())

Training {0: 1373, 1: 346, 2: 1476}
Validation {0: 344, 1: 86, 2: 369}
Test {0: 430, 1: 108, 2: 461}


In [41]:
# End-to-end CNN for classifying galaxies into three categories

class GalaxyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Extract visual features from the galaxy images
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Use the extracted features to predict one of three classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [75]:
# GPU

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

Using: cuda
NVIDIA GeForce RTX 5070 Laptop GPU


In [74]:
# Reset the random seed before initialising the baseline CNN
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Create a fresh model with randomly initialised weights
model = GalaxyCNN().to(device)

# Set up the same loss function and optimizer as before
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("End-to-end CNN reset and ready for training")
print(model)

End-to-end CNN reset and ready for training
GalaxyCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [45]:
# Get one batch of galaxy images
images, labels = next(iter(train_loader))

# Move the images to the same device as the CNN
images = images.to(device)

# Run the images through the CNN
with torch.no_grad():
    outputs = model(images)

print("Output shape:", outputs.shape)
print("Label shape:", labels.shape)

Output shape: torch.Size([32, 3])
Label shape: torch.Size([32])


## End-to-End CNN Training and Validation

In [76]:
# Train the three-class end-to-end CNN

num_epochs = 10

train_losses = []
validation_losses = []

for epoch in range(num_epochs):
    # Training mode enables weight updates
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Predict class scores and calculate the error
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Calculate gradients and update model weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # Evaluate on validation data without updating weights
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / len(validation_dataset)
    validation_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Training loss: {epoch_train_loss:.4f} | "
        f"Validation loss: {epoch_val_loss:.4f}"
    )

Epoch 1/10 | Training loss: 0.9273 | Validation loss: 0.8701
Epoch 2/10 | Training loss: 0.7841 | Validation loss: 0.7187
Epoch 3/10 | Training loss: 0.7078 | Validation loss: 0.6557
Epoch 4/10 | Training loss: 0.6418 | Validation loss: 0.6238
Epoch 5/10 | Training loss: 0.6291 | Validation loss: 0.6301
Epoch 6/10 | Training loss: 0.6016 | Validation loss: 0.6186
Epoch 7/10 | Training loss: 0.5700 | Validation loss: 0.6126
Epoch 8/10 | Training loss: 0.5522 | Validation loss: 0.6001
Epoch 9/10 | Training loss: 0.5490 | Validation loss: 0.5849
Epoch 10/10 | Training loss: 0.5153 | Validation loss: 0.6227


In [77]:
# Evaluate the end-to-end CNN on validation data

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in validation_loader:
        images = images.to(device)

        # Predict the most likely class for each image
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

class_names = [
    "Smooth",
    "Featured edge-on",
    "Featured not edge-on"
]

# Overall accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Validation accuracy: {accuracy:.4f}")

# Precision, recall and F1-score for each class
print("\nClassification report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2],
        target_names=class_names,
        zero_division=0
    )
)

# Show which classes are being confused
print("\nConfusion matrix:")
print(confusion_matrix(all_labels, all_predictions, labels=[0, 1, 2]))

Validation accuracy: 0.7334

Classification report:
                      precision    recall  f1-score   support

              Smooth       0.73      0.80      0.76       344
    Featured edge-on       0.60      0.81      0.69        86
Featured not edge-on       0.79      0.65      0.71       369

            accuracy                           0.73       799
           macro avg       0.71      0.76      0.72       799
        weighted avg       0.74      0.73      0.73       799


Confusion matrix:
[[276  13  55]
 [  7  70   9]
 [ 95  34 240]]


## Hierarchical CNN

In [51]:
# Hierarchical CNN with separate heads for Galaxy Zoo Tasks 01 and 02

class HierarchicalGalaxyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Same convolutional feature extractor as the baseline CNN
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Shared fully connected layer
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )

        # Task 01: Smooth (0) or featured (1)
        self.task1_head = nn.Linear(128, 2)

        # Task 02: Edge-on (0) or not edge-on (1)
        self.task2_head = nn.Linear(128, 2)

    def forward(self, x):
        # Extract visual features once for both tasks
        x = self.features(x)
        x = self.shared(x)

        # Produce predictions for both classification decisions
        task1_outputs = self.task1_head(x)
        task2_outputs = self.task2_head(x)

        return task1_outputs, task2_outputs

In [78]:
# Reset the random seed before initialising the hierarchical CNN
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Create a fresh hierarchical CNN
hierarchical_model = HierarchicalGalaxyCNN().to(device)

# Set up the hierarchical training loss and optimizer
hierarchical_criterion = nn.CrossEntropyLoss()

hierarchical_optimizer = optim.Adam(
    hierarchical_model.parameters(),
    lr=0.001
)

print("Hierarchical CNN reset and ready for training")
print(hierarchical_model)

Hierarchical CNN reset and ready for training
HierarchicalGalaxyCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (shared): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
  )
  (task1_head): Linear(in_features=128, out_features=2, bias=True)
  (task2_head): Linear(in_features=128, out_features=2, bias=True)
)


In [53]:
# Check that both heads produce two scores per galaxy
images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    task1_outputs, task2_outputs = hierarchical_model(images)

print("Task 01 output shape:", task1_outputs.shape)
print("Task 02 output shape:", task2_outputs.shape)
print("Label shape:", labels.shape)

Task 01 output shape: torch.Size([32, 2])
Task 02 output shape: torch.Size([32, 2])
Label shape: torch.Size([32])


In [79]:
# Train and validate the hierarchical CNN

num_epochs = 10

hierarchical_train_losses = []
hierarchical_validation_losses = []

for epoch in range(num_epochs):
    hierarchical_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Task 01: 0 = smooth, 1 = featured
        task1_labels = (labels != 0).long()

        # Task 02 only applies to featured galaxies
        featured_mask = labels != 0

        # Task 02: 0 = edge-on, 1 = not edge-on
        task2_labels = (labels[featured_mask] == 2).long()

        hierarchical_optimizer.zero_grad()

        task1_outputs, task2_outputs = hierarchical_model(images)

        # Every galaxy contributes to Task 01
        task1_loss = hierarchical_criterion(
            task1_outputs, task1_labels
        )

        # Only featured galaxies contribute to Task 02
        if featured_mask.any():
            task2_loss = hierarchical_criterion(
                task2_outputs[featured_mask],
                task2_labels
            )
        else:
            task2_loss = torch.tensor(0.0, device=device)

        # Combine the two classification losses
        loss = task1_loss + task2_loss

        loss.backward()
        hierarchical_optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    hierarchical_train_losses.append(epoch_train_loss)

    # Validate without updating model weights
    hierarchical_model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            task1_labels = (labels != 0).long()
            featured_mask = labels != 0
            task2_labels = (labels[featured_mask] == 2).long()

            task1_outputs, task2_outputs = hierarchical_model(images)

            task1_loss = hierarchical_criterion(
                task1_outputs, task1_labels
            )

            if featured_mask.any():
                task2_loss = hierarchical_criterion(
                    task2_outputs[featured_mask],
                    task2_labels
                )
            else:
                task2_loss = torch.tensor(0.0, device=device)

            loss = task1_loss + task2_loss

            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / len(validation_dataset)
    hierarchical_validation_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Training loss: {epoch_train_loss:.4f} | "
        f"Validation loss: {epoch_val_loss:.4f}"
    )

Epoch 1/10 | Training loss: 1.1250 | Validation loss: 1.0190
Epoch 2/10 | Training loss: 0.9387 | Validation loss: 0.8679
Epoch 3/10 | Training loss: 0.8155 | Validation loss: 0.7993
Epoch 4/10 | Training loss: 0.7788 | Validation loss: 0.7678
Epoch 5/10 | Training loss: 0.7594 | Validation loss: 0.7451
Epoch 6/10 | Training loss: 0.7377 | Validation loss: 0.7919
Epoch 7/10 | Training loss: 0.6972 | Validation loss: 0.7362
Epoch 8/10 | Training loss: 0.7029 | Validation loss: 0.8163
Epoch 9/10 | Training loss: 0.6781 | Validation loss: 0.7846
Epoch 10/10 | Training loss: 0.6605 | Validation loss: 0.7342


In [80]:
# Evaluate hierarchical CNN on the validation set

hierarchical_model.eval()

hierarchical_predictions = []
hierarchical_labels = []

with torch.no_grad():
    for images, labels in validation_loader:
        images = images.to(device)

        # Get predictions from both decision heads
        task1_outputs, task2_outputs = hierarchical_model(images)

        task1_predictions = torch.argmax(task1_outputs, dim=1)
        task2_predictions = torch.argmax(task2_outputs, dim=1)

        # Combine the decisions into the three final classes
        # Smooth = 0, featured edge-on = 1, featured not edge-on = 2
        final_predictions = torch.where(
            task1_predictions == 0,
            0,
            task2_predictions + 1
        )

        hierarchical_predictions.extend(
            final_predictions.cpu().numpy()
        )
        hierarchical_labels.extend(labels.numpy())

# Calculate validation metrics
hierarchical_accuracy = accuracy_score(
    hierarchical_labels,
    hierarchical_predictions
)

print(f"Hierarchical validation accuracy: {hierarchical_accuracy:.4f}")

print("\nClassification report:")
print(
    classification_report(
        hierarchical_labels,
        hierarchical_predictions,
        labels=[0, 1, 2],
        target_names=class_names,
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        hierarchical_labels,
        hierarchical_predictions,
        labels=[0, 1, 2]
    )
)

Hierarchical validation accuracy: 0.7146

Classification report:
                      precision    recall  f1-score   support

              Smooth       0.83      0.60      0.70       344
    Featured edge-on       0.58      0.79      0.67        86
Featured not edge-on       0.68      0.80      0.74       369

            accuracy                           0.71       799
           macro avg       0.70      0.73      0.70       799
        weighted avg       0.74      0.71      0.71       799


Confusion matrix:
[[208  16 120]
 [  2  68  16]
 [ 41  33 295]]


In [81]:
# Analyse where hierarchical classification errors occur

true_labels = np.array(hierarchical_labels)
predicted_labels = np.array(hierarchical_predictions)

# Task 01 error: Smooth vs featured decision is incorrect
task1_errors = (true_labels == 0) != (predicted_labels == 0)

# Task 02 error: Both labels are featured, but edge-on decision is incorrect
task2_errors = (
    (true_labels != 0) &
    (predicted_labels != 0) &
    (true_labels != predicted_labels)
)

total_errors = np.sum(true_labels != predicted_labels)

print(f"Total classification errors: {total_errors}")
print(f"Task 01 errors: {np.sum(task1_errors)}")
print(f"Task 02 errors: {np.sum(task2_errors)}")

print(
    f"Percentage of errors from Task 01: "
    f"{np.sum(task1_errors) / total_errors * 100:.1f}%"
)

print(
    f"Percentage of errors from Task 02: "
    f"{np.sum(task2_errors) / total_errors * 100:.1f}%"
)

Total classification errors: 228
Task 01 errors: 179
Task 02 errors: 49
Percentage of errors from Task 01: 78.5%
Percentage of errors from Task 02: 21.5%


In [82]:
# Oracle-routing experiment: use the correct Task 01 decision
# while keeping Task 02 predictions unchanged

hierarchical_model.eval()

oracle_predictions = []
oracle_labels = []

with torch.no_grad():
    for images, labels in validation_loader:
        images = images.to(device)

        # Obtain predictions from the trained hierarchical model
        task1_outputs, task2_outputs = hierarchical_model(images)
        task2_predictions = torch.argmax(task2_outputs, dim=1)

        # Use the true label to determine whether a galaxy is featured
        true_featured = labels.to(device) != 0

        # Smooth galaxies remain class 0
        # Featured galaxies use the model's Task 02 prediction
        final_predictions = torch.where(
            true_featured,
            task2_predictions + 1,
            0
        )

        oracle_predictions.extend(final_predictions.cpu().numpy())
        oracle_labels.extend(labels.numpy())

oracle_accuracy = accuracy_score(
    oracle_labels,
    oracle_predictions
)

print(f"Normal hierarchical accuracy: {hierarchical_accuracy:.4f}")
print(f"Oracle-routing accuracy: {oracle_accuracy:.4f}")
print(
    f"Accuracy improvement: "
    f"{(oracle_accuracy - hierarchical_accuracy) * 100:.2f} percentage points"
)

Normal hierarchical accuracy: 0.7146
Oracle-routing accuracy: 0.9337
Accuracy improvement: 21.90 percentage points


In [59]:
# Compare the number of correct predictions with normal and oracle routing
normal_correct = np.sum(
    np.array(hierarchical_predictions) == np.array(hierarchical_labels)
)

oracle_correct = np.sum(
    np.array(oracle_predictions) == np.array(oracle_labels)
)

print(f"Normal correct predictions: {normal_correct}")
print(f"Oracle correct predictions: {oracle_correct}")
print(f"Additional correct predictions: {oracle_correct - normal_correct}")

Normal correct predictions: 602
Oracle correct predictions: 755
Additional correct predictions: 153


In [60]:
# Check whether flat and hierarchical label assignment agree

# Existing three-class labels
flat_targets = classification_data["target"].to_numpy()

# Determine Task 01 using its original smooth/featured probabilities
smooth = classification_data["Class1.1"].to_numpy()
featured = classification_data["Class1.2"].to_numpy()

# Determine Task 02 using its original edge-on probabilities
edge_on = classification_data["Class2.1"].to_numpy()
not_edge_on = classification_data["Class2.2"].to_numpy()

# Assign labels by following the Galaxy Zoo decision tree
hierarchical_targets = np.where(
    smooth >= featured,
    0,
    np.where(edge_on >= not_edge_on, 1, 2)
)

different_labels = np.sum(flat_targets != hierarchical_targets)

print(f"Total galaxies: {len(flat_targets)}")
print(f"Different labels: {different_labels}")
print(
    f"Percentage different: "
    f"{different_labels / len(flat_targets) * 100:.2f}%"
)

Total galaxies: 4993
Different labels: 109
Percentage different: 2.18%


In [61]:
# Examine how the original and hierarchical labels disagree
from sklearn.metrics import confusion_matrix

print("Rows: original labels")
print("Columns: hierarchical labels")

print(
    confusion_matrix(
        flat_targets,
        hierarchical_targets,
        labels=[0, 1, 2]
    )
)

Rows: original labels
Columns: hierarchical labels
[[2147   29   80]
 [   0  511    0]
 [   0    0 2226]]
